# Name That LLM — submission notebook

18-class attribution with sparse stylometry plus **crossed-design inference**
(D016). **Offline by design**: no network calls, no downloads, no API keys.

Inputs required on the right-hand panel:
1. the competition dataset (`name-that-llm`), and
2. the prediction-artifacts dataset built with
   `python -m src.cli package --crossed --kaggle-user <your-username>`. It contains
   `manifest.json`, `models/`, `weights.json`, `inference.json`, the MiniLM
   grouping encoder (`encoder/minilm`), and the `src/` package needed to unpickle
   the bundles.

Inference: blend the char + style margins, cluster the test passages into prompt
groups with the attached MiniLM encoder (cosine distance 0.4), then enforce
one-passage-per-model assignment within each group. If `inference.json` is absent
the notebook falls back to plain argmax blending.

Runtime dependency: `sentence-transformers` (+ `transformers`, `torch`) must be
importable in the Kaggle image to load the grouping encoder; the weights
themselves are attached, never downloaded.

Output: `submission.parquet` in the working directory, schema `id, model`.

In [ ]:
from __future__ import annotations

import glob
import json
import os
import sys
from pathlib import Path

# Kaggle inference has no internet. Force offline mode so the attached grouping
# encoder loads from disk instead of attempting a download.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

import pandas as pd

ARTIFACT_CANDIDATES = ([str(Path(os.environ["NTL_ARTIFACT_DIR"]) / "manifest.json")]
    if os.environ.get("NTL_ARTIFACT_DIR") else
    glob.glob("/kaggle/input/**/manifest.json", recursive=True))
if not ARTIFACT_CANDIDATES:
    raise FileNotFoundError(
        "No manifest.json found under /kaggle/input. Attach the artifacts dataset "
        "created by `python -m src.cli package` (add --crossed for the crossed pipeline)."
    )
if len(ARTIFACT_CANDIDATES) != 1:
    raise ValueError(f"Expected one artifact manifest, found {len(ARTIFACT_CANDIDATES)}; attach only the intended dataset")
ARTIFACT_DIR = Path(ARTIFACT_CANDIDATES[0]).parent
sys.path.insert(0, str(ARTIFACT_DIR))          # lets `src` be importable
sys.path.insert(0, str(ARTIFACT_DIR / "src"))  # fallback layout

from src.constants import LABELS, N_CLASSES  # noqa: E402
from src.inference import (  # noqa: E402
    load_final,
    predict_dataframe,
    predict_dataframe_crossed,
    validate_weight_provenance,
)
from src.submission import build_submission  # noqa: E402

print(f"artifacts: {ARTIFACT_DIR}")
print(f"classes: {N_CLASSES}")

In [ ]:
TEST_PATHS = ([Path(os.environ["NTL_TEST_PATH"])] if os.environ.get("NTL_TEST_PATH") else []) + [
    Path("/kaggle/input/competitions/name-that-llm/test.csv"),
    Path("/kaggle/input/name-that-llm/test.csv"),
    Path("data/synthetic/test.csv"),  # local smoke test only
]
test_path = next((p for p in TEST_PATHS if p.is_file()), None)
if test_path is None:
    raise FileNotFoundError("could not locate test.csv in any known input path")

test = pd.read_csv(test_path)
assert {"id", "text"} <= set(test.columns), f"unexpected columns: {list(test.columns)}"
assert test["id"].is_unique, "test.csv has duplicate ids"
assert test["text"].notna().all(), "test.csv has null text"
print(f"test rows: {len(test)} from {test_path}")

In [ ]:
bundles, manifest = load_final(ARTIFACT_DIR)
assert manifest["class_order"] == list(LABELS), "artifact class order mismatch"

weights_path = ARTIFACT_DIR / "weights.json"
weights = None
if weights_path.is_file():
    weights = validate_weight_provenance(weights_path, manifest, bundles)
elif len(bundles) > 1:
    raise ValueError("Multiple models require verified weights.json; rebuild the package")

print("models:", sorted(bundles))
print("blend weights:", weights or "equal weights")

inference_path = ARTIFACT_DIR / "inference.json"
inference = (
    json.loads(inference_path.read_text(encoding="utf-8"))
    if inference_path.is_file()
    else {"crossed": False}
)

if inference.get("crossed"):
    # Crossed-design inference (D016): cluster passages into prompt groups with
    # the attached MiniLM encoder, then enforce one-passage-per-model assignment
    # within each recovered group. Label-free and transductive (test text only).
    from src.features_embedding import FrozenEncoder  # noqa: E402

    encoder_dir = ARTIFACT_DIR / inference["encoder_path"]
    if not encoder_dir.is_dir():
        raise FileNotFoundError(
            f"crossed inference needs the grouping encoder at {encoder_dir}; "
            "rebuild the package with `python -m src.cli package --crossed`."
        )
    encoder = FrozenEncoder(
        backend=inference.get("encoder_backend", "sentence-transformers"),
        model_name=str(encoder_dir),
    )
    print(
        f"crossed inference: encoder={encoder_dir.name}, "
        f"distance_threshold={inference.get('distance_threshold')}"
    )
    predictions = predict_dataframe_crossed(
        bundles, test, weights, encoder=encoder,
        distance_threshold=inference.get("distance_threshold", 0.4),
        n_clusters=inference.get("n_clusters"),
    )
else:
    predictions = predict_dataframe(bundles, test, weights)

assert len(predictions) == len(test)
assert set(predictions) <= set(LABELS)
# Sanity: the crossed design yields a near-uniform label distribution (~1/18 each).
print(pd.Series(predictions).value_counts().head())

In [ ]:
out = build_submission(
    test["id"], predictions, "submission.parquet", expected_ids=test["id"]
)
check = pd.read_parquet(out)
assert list(check.columns) == ["id", "model"]
assert len(check) == len(test)
assert check["id"].is_unique and not check.isna().any().any()
assert check["model"].isin(LABELS).all()
print(f"wrote {out} with {len(check)} rows")
check.head()

## Notes

- Internet is disabled for this competition; the notebook never fetches weights,
  models or data at runtime. `HF_HUB_OFFLINE`/`TRANSFORMERS_OFFLINE` are set so the
  attached MiniLM encoder loads from disk.
- `submission.parquet` is written to the working directory, as required.
- The class order is fixed by the artifact manifest and checked against
  `src.constants.LABELS`; a mismatched artifact fails loudly instead of silently
  permuting labels.
- **Crossed inference (D016)** is label-free and transductive: it clusters test
  passages by topic to recover prompt groups, then assigns one passage per model
  within each group. This exploits the balanced full-factorial design and was
  evaluated on a historical secondary split (+0.104) and alternative fold seeds;
  these results are exploratory, not an independent holdout. A
  healthy run shows a near-uniform label distribution (~1/18 per class); a
  degenerate distribution would signal clustering collapse on the real test set.
- The grouping encoder is the frozen `all-MiniLM-L6-v2` checkpoint (Apache-2.0),
  attached offline. It is used only for clustering, not for authorship features.
- If this notebook is run on the synthetic fixture (`data/synthetic/test.csv`),
  the output is a pipeline check, not a competition submission.